In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [3]:
import os
from datetime import datetime
from anthropic import Anthropic
from anthropic.types import ToolParam
from dotenv import load_dotenv

load_dotenv()

anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
anthropic_model = "claude-3-5-haiku-latest"


# 1. Python Function
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


# 2. Anthropic Schema Definition
get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Returns the current date and time as a formatted string. Use this when the user asks for the current time, today's date, or needs a timestamp.",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": 'Python strftime format string for the output. Common formats: "%Y-%m-%d %H:%M:%S" (full datetime), "%Y-%m-%d" (date only), "%H:%M:%S" (time only), "%B %d, %Y" (e.g. January 15, 2026). Must not be empty.',
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }
)

# 3. Call with Anthropic SDK
messages = [
    {"role": "user", "content": "What is the exact time, formatted as HH:MM:SS?"}
]

response = anthropic_client.messages.create(
    model=anthropic_model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

messages.append({
  "role": "assistant",
    "content": response.content
})

# Inspect tool calls generated by the model
print(response.content)

In [ ]:
result = get_current_datetime(**response.content[1].input)

In [ ]:
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_user_id": response.content[1].id,
            "content": result,
            "is_error": False
        }
    ]
})

messages

In [ ]:
response = anthropic_client.messages.create(
    model=anthropic_model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)


In [12]:
import os
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    base_url="https://api.aicredits.in/v1",
    api_key=os.getenv("AICREDITS_API_KEY"),
)
openai_model = "anthropic/claude-haiku-4-5"


# 1. Python Function
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


# 2. OpenAI Tool Definition
get_current_datetime_tool = {
    "type": "function",
    "function": {
        "name": "get_current_datetime",
        "description": "Returns the current date and time as a formatted string. Use this when the user asks for the current time, today's date, or needs a timestamp.",
        "parameters": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": 'Python strftime format string for the output. Common formats: "%Y-%m-%d %H:%M:%S" (full datetime), "%Y-%m-%d" (date only), "%H:%M:%S" (time only), "%B %d, %Y" (e.g. January 15, 2026). Must not be empty.',
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    },
}

# 3. Call with OpenAI SDK
messages = [
    {"role": "user", "content": "What is the exact time, formatted as HH:MM:SS?"}
]

response = openai_client.chat.completions.create(
    model=openai_model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_tool],
)

# Append assistant message with tool call details
messages.append(response.choices[0].message)



print("message:",messages)

print("--------")

print("response:", response)

print("--------")

# Inspect tool calls generated by the model
tool_calls = response.choices[0].message.tool_calls
print(tool_calls)

message: [{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='toolu_012cRkKpXcdtgNHCThhEsKuA', function=Function(arguments='{"date_format":"%H:%M:%S"}', name='get_current_datetime'), type='function', index=0)])]
--------
response: ChatCompletion(id='chatcmpl-claude', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='toolu_012cRkKpXcdtgNHCThhEsKuA', function=Function(arguments='{"date_format":"%H:%M:%S"}', name='get_current_datetime'), type='function', index=0)]), content_filter_results={'hate': {'filtered': False}, 'jailbreak': {'detected': False, 'filtered': False}, 'profanity': {'detected': Fal

In [13]:
import json

# Extract tool call
tool_call = response.choices[0].message.tool_calls[0]

# OpenAI returns arguments as a JSON string -> must parse to dict
args = json.loads(tool_call.function.arguments)

# Execute function
result = get_current_datetime(**args)
print(result)

17:07:18


### Difference for OpenAISDK tool call and Anthropic SDK tool call

| Feature / Step | OpenAI SDK (via [AICredits](https://aicredits.in/)) | Native Anthropic SDK |
| --- | --- | --- |
| **Tool Definition Schema** | `{"type": "function", "function": {"parameters": {...}}}` | `ToolParam({"input_schema": {...}})` |
| **Request Parameter** | `tools=[get_current_datetime_tool]` | `tools=[get_current_datetime_schema]` |
| **Response Location** | `response.choices[0].message.tool_calls` | `response.content` |
| **Tool Block Identification** | `tool_calls` list on message object | Item with `block.type == "tool_use"` |
| **Function Name** | `tool_call.function.name` | `tool_block.name` |
| **Function Arguments** | `tool_call.function.arguments` (JSON **`str`**) | `tool_block.input` (Python **`dict`**) |
| **Execution Call** | `get_current_datetime(**json.loads(tool_call.function.arguments))` | `get_current_datetime(**response.content[1].input)` |
| **Tool Call ID** | `tool_call.id` | `tool_block.id` |
| **Returning Result to Context** | `{"role": "tool", "tool_call_id": tool_call.id, "content": str(result)}` | `{"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": str(result)}]}` |

In [14]:
# 1. Append the tool execution result using role: "tool"
messages.append(
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(result),
    }
)

# 2. Complete the turn to get Claude's final answer
final_response = openai_client.chat.completions.create(
    model=openai_model,
    messages=messages,
)

print(final_response.choices[0].message.content)

The exact time is **17:07:18** (5:07:18 PM)


### Complete Function Calling Lifecycle: OpenAI SDK vs Anthropic SDK

---

#### Step 1: Define the Python Function
Define the Python function to be executed locally on your machine.

```python
from datetime import datetime

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

```

---

#### Step 2: Declare the Tool Schema

Define the tool schema so the model understands when and how to call it.

* **OpenAI SDK (via AICredits):** Uses `parameters` wrapped inside a `function` dictionary.

```python
get_current_datetime_tool = {
    "type": "function",
    "function": {
        "name": "get_current_datetime",
        "description": "Returns current datetime as a formatted string.",
        "parameters": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "Python strftime format string.",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": []
        }
    }
}

```

* **Native Anthropic SDK:** Uses `input_schema` directly on the tool definition.

```python
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns current datetime as a formatted string.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "Python strftime format string.",
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": []
    }
})

```

---

#### Step 3: Trigger the Tool Call Request

Send the user query alongside the tool schema.

* **OpenAI SDK (via AICredits):**

```python
messages = [{"role": "user", "content": "What is the exact time, formatted as HH:MM:SS?"}]

response = openai_client.chat.completions.create(
    model=openai_model,
    messages=messages,
    tools=[get_current_datetime_tool]
)

```

* **Native Anthropic SDK:**

```python
messages = [{"role": "user", "content": "What is the exact time, formatted as HH:MM:SS?"}]

response = anthropic_client.messages.create(
    model=anthropic_model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

```

---

#### Step 4: Record Model Decision into Conversation History

Log the model's tool request before fulfilling it.

* **OpenAI SDK (via AICredits):**

```python
# Append the assistant response object directly to retain tool_calls
messages.append(response.choices[0].message)

```

* **Native Anthropic SDK:**

```python
# Append the assistant message containing content blocks (including type='tool_use')
messages.append({"role": "assistant", "content": response.content})

```

---

#### Step 5: Parse Arguments & Execute Local Code

Extract the arguments sent by the model and invoke the function.

* **OpenAI SDK (via AICredits):** Arguments come as a raw JSON string and must be parsed with `json.loads()`.

```python
import json

tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
result = get_current_datetime(**args)

```

* **Native Anthropic SDK:** Arguments are pre-parsed into a Python dictionary.

```python
tool_block = next(b for b in response.content if b.type == "tool_use")
result = get_current_datetime(**tool_block.input)

```

---

#### Step 6: Return the Tool Output

Feed the tool result back into the message list.

* **OpenAI SDK (via AICredits):** Uses a dedicated message with `role: "tool"`.

```python
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": str(result)
})

```

* **Native Anthropic SDK:** Uses a message with `role: "user"` containing a `tool_result` content block.

```python
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_block.id,
        "content": str(result)
    }]
})

```

---

#### Step 7: Final Request for Natural Language Output

Send the full conversation history to produce the final user-facing response.

* **OpenAI SDK (via AICredits):**

```python
final_response = openai_client.chat.completions.create(
    model=openai_model,
    messages=messages
)
print(final_response.choices[0].message.content)

```

* **Native Anthropic SDK:**

```python
final_response = anthropic_client.messages.create(
    model=anthropic_model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)
print(final_response.content[0].text)

```

```

```